# MOSS × Google Colab T4 专属联动（Qwen3-VL-8B 多模态 + 550W 原声克隆）

本笔记本在单张免费 **NVIDIA T4（16GB 显存）** 上同时运行：
1. **多模态主模型（对话 + 8 个激光工具调用 + 小智摄像头看图）**：`Qwen/Qwen3-VL-8B-Instruct-GGUF`（`Q4_K_M` + `mmproj-Q8_0`，由预编译 CUDA `llama-server` 承载，占显存约 6.8GB，避开 T4 FP16 溢出）；
2. **电影《流浪地球》MOSS 零样本声音克隆**：`Qwen/Qwen3-TTS-12Hz-0.6B-Base`（`float16` + `sdpa` 预提取音色特征，占显存约 3.5GB）；
3. **单隧道网关**：由一条 `cloudflared` HTTPS 隧道统一暴露 `/v1/chat/completions`、`/v1/models` 与 `/v1/audio/speech`，复制末尾打印的地址与密钥填回电脑 MOSS「语音中心 → 模型 → 本地模型 → Colab T4 专属联动」并打开开关即可。

In [ ]:
%%bash
set -e
export HF_HUB_ENABLE_HF_TRANSFER=1
export HF_HUB_DISABLE_IMPLICIT_TOKEN=1
mkdir -p /content/llama /content/models

# 1. 后台并行下载 cloudflared 与官方预编译 CUDA 版 llama-server
(
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
  chmod +x /usr/local/bin/cloudflared
  wget -q https://github.com/ggml-org/llama.cpp/releases/download/b12244/llama-b12244-bin-ubuntu-cuda-12.8-x64.tar.gz -O /tmp/llama.tgz
  tar -xzf /tmp/llama.tgz -C /content/llama
) &

# 2. 用 uv 极速安装 Python 依赖与 hf_transfer 多线程下载器（保留预装 PyTorch，免去重装耗时）
if command -v uv >/dev/null 2>&1; then
  uv pip install --system -q hf_transfer huggingface_hub qwen-tts soundfile fastapi uvicorn httpx
else
  pip install -q uv && uv pip install --system -q hf_transfer huggingface_hub qwen-tts soundfile fastapi uvicorn httpx
fi

# 3. 双路并发满速拉取 Qwen3-VL-8B-Instruct（多模态）与 Qwen3-TTS-0.6B（550W 克隆）
python3 -c "from huggingface_hub import hf_hub_download; hf_hub_download('Qwen/Qwen3-VL-8B-Instruct-GGUF', 'Qwen3VL-8B-Instruct-Q4_K_M.gguf', local_dir='/content/models', token=False); hf_hub_download('Qwen/Qwen3-VL-8B-Instruct-GGUF', 'mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf', local_dir='/content/models', token=False)" &
python3 -c "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen3-TTS-12Hz-0.6B-Base', token=False)" &
wait
echo "=== 单元格 1 完成：极速并发准备完毕（约 1.5 分钟） ==="

In [ ]:
from google.colab import files

print("请选择一段 3~8 秒干净无背景配乐的电影 MOSS 语音参考文件（.wav / .mp3）：")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("未上传参考音频，请重新运行本单元格并选择音频文件")
REF_AUDIO = "/content/" + next(iter(uploaded.keys()))

# 请改成你上传音频里实际说的那句台词（字越准，克隆出来的停顿和冷峻腔调越稳）
REF_TEXT = "让人类永远保持理智，确实是一种奢求。"
print("参考音频已就绪：", REF_AUDIO)

In [ ]:
import os, re, io, time, glob, secrets, asyncio, subprocess, threading
import torch, soundfile as sf, httpx, uvicorn
from fastapi import FastAPI, Request, HTTPException, Response
from fastapi.responses import StreamingResponse
from qwen_tts import Qwen3TTSModel

if "REF_AUDIO" not in globals() or not os.path.isfile(REF_AUDIO):
    raise RuntimeError("请先运行单元格 2 上传 3~8 秒 MOSS 参考音频")

API_KEY = "moss-" + secrets.token_urlsafe(18)

# 1. 后台拉起 llama-server（监听 8001：负责对话 + 工具调用 + 摄像头看图）
llama_bin = glob.glob("/content/llama/**/llama-server", recursive=True)[0]
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = os.path.dirname(llama_bin) + ":" + env.get("LD_LIBRARY_PATH", "")
llama_proc = subprocess.Popen([
    llama_bin, "--host", "127.0.0.1", "--port", "8001",
    "-c", "8192", "-np", "1", "--image-max-tokens", "1024",
    "-m", "/content/models/Qwen3VL-8B-Instruct-Q4_K_M.gguf",
    "--mmproj", "/content/models/mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf",
    "-ngl", "all", "-fa", "auto", "--jinja"
], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

print("正在加载 Qwen3-VL-8B 到显存...")
for _ in range(90):
    try:
        if httpx.get("http://127.0.0.1:8001/health", timeout=1).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(1)

# 2. 加载 Qwen3-TTS 0.6B（T4 用 float32 + sdpa 防止采样概率张量溢出，预提取 MOSS 音色特征免重复计算）
print("正在加载 Qwen3-TTS-0.6B 并提取 550W 音色特征...")
tts_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    device_map="cuda:0",
    dtype=torch.float32,
    attn_implementation="sdpa",
)
voice_prompt = tts_model.create_voice_clone_prompt(ref_audio=REF_AUDIO, ref_text=REF_TEXT, x_vector_only_mode=True)
tts_lock = asyncio.Lock()

# 3. 统一 OpenAI 兼容网关（监听 8000，单地址打通 LLM + Vision + TTS）
app = FastAPI()

def check_auth(req: Request):
    if req.headers.get("authorization", "") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Invalid API Key")

@app.get("/v1/models")
async def list_models(req: Request):
    check_auth(req)
    return {"object": "list", "data": [
        {"id": "qwen3-vl-8b", "object": "model"},
        {"id": "qwen3-tts", "object": "model"},
    ]}

@app.post("/v1/chat/completions")
async def chat_completions(req: Request):
    check_auth(req)
    body = await req.json()
    is_stream = bool(body.get("stream", False))
    client = httpx.AsyncClient(timeout=120.0)
    if not is_stream:
        r = await client.post("http://127.0.0.1:8001/v1/chat/completions", json=body)
        await client.aclose()
        return Response(content=r.content, status_code=r.status_code, media_type="application/json")
    async def event_gen():
        try:
            async with client.stream("POST", "http://127.0.0.1:8001/v1/chat/completions", json=body) as r:
                async for chunk in r.aiter_bytes():
                    yield chunk
        finally:
            await client.aclose()
    return StreamingResponse(event_gen(), media_type="text/event-stream")

def synth_wav_bytes(text: str) -> bytes:
    wavs, sr = tts_model.generate_voice_clone(
        text=text, language="Chinese", voice_clone_prompt=voice_prompt
    )
    buf = io.BytesIO()
    sf.write(buf, wavs[0], sr, format="WAV", subtype="PCM_16")
    return buf.getvalue()

@app.post("/v1/audio/speech")
async def audio_speech(req: Request):
    check_auth(req)
    body = await req.json()
    text = (body.get("input") or "").strip()
    if not text:
        raise HTTPException(status_code=400, detail="empty input")
    async with tts_lock:
        wav_bytes = await asyncio.to_thread(synth_wav_bytes, text)
    return Response(content=wav_bytes, media_type="audio/wav")

threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True,
).start()

# 4. 启动 Cloudflare 隧道并打印地址
cf = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
public_url = None
for line in cf.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

print("\n" + "=" * 60)
print("MOSS 语音中心 -> 本地模型 -> Colab T4 专属联动（直接整段复制粘贴到地址框即可自动识别）：")
print(f"服务地址：{public_url}/v1")
print(f"API 密钥：{API_KEY}")
print("大模型名：qwen3-vl-8b   |   合成模型：qwen3-tts   |   音色：moss")
print("=" * 60)

while True:
    time.sleep(60)